In [1]:
import torch
print(torch.cuda.is_available())

False


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!ls /content/drive/MyDrive/

 20220708_152317.jpg
'2023-11-07 Statement - USB Checking 7348.pdf'
 2534868b36f74bbda0d0479c9f074d46.MOV
 34043_COMBINED_6302_29042021.gdoc
'4th year result.jpg'
 73546835e2ac43ecb685f18ce393f6e4.MOV
'Applied AI Summit Topic.gdoc'
'Article 1.gdoc'
'Article 2.gdoc'
'Article 3.gdoc'
'Articles Summary.gdoc'
'Assignment 11.docx'
'B2 Visa Interview Questions (1).gdoc'
'B2 Visa Interview Questions.gdoc'
 bq-results-20260307-234234-1772926980937
 bq-results-20260307-234413-1772927084661
 clinicalbert_results.txt
'Colab Notebooks'
'Copy of Copy of Copy of CSAI Week update V1.5- our interns.gslides'
'Copy of Copy of CSAI Week update V1.4- our interns.gslides'
'Copy of Copy of CSAI Week update V1.5- our interns.gslides'
'Copy of CSAI Week update V1.1- RWE.gslides'
'Copy of CSAI Week update V1.3- our interns.gslides'
'Copy of [Open] Applied AI Summit Speaker Slide Tempalte - 2025 v2.gslides'
'Copy of RWE Intern slide-1.9.gslides'
'Cover Letter.pdf'
'CSAI RWE template.gform'
'CSAI Week update V1.

In [4]:
!ls /content/

drive  sample_data


In [5]:
!ls /content/data/

ls: cannot access '/content/data/': No such file or directory


In [8]:
!ls /content/data/in-hospital-mortality-cleaned/train | head

10004_episode1_timeseries.csv
10007_episode1_timeseries.csv
1000_episode1_timeseries.csv
10013_episode1_timeseries.csv
10017_episode1_timeseries.csv
10027_episode1_timeseries.csv
10028_episode1_timeseries.csv
10029_episode1_timeseries.csv
10032_episode1_timeseries.csv
10038_episode1_timeseries.csv


In [9]:
!find /content/data -name "*listfile*"

In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


In [11]:
!rm -rf /content/data

In [2]:
!unzip "/content/drive/MyDrive/mimic_clean_for_colab.zip" -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/in-hospital-mortality-cleaned/train/71479_episode1_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/77625_episode1_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/28065_episode3_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/11342_episode2_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/82065_episode1_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/18219_episode1_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/96259_episode2_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/23014_episode2_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/48523_episode1_timeseries.csv  
  inflating: /content/data/in-hospital-mortality-cleaned/train/12899_episode1_timeseries.csv  

In [7]:
!ls /content/data/in-hospital-mortality
!ls /content/data/in-hospital-mortality-cleaned/train | head

test  test_listfile.csv  train	train_listfile.csv  val_listfile.csv
10004_episode1_timeseries.csv
10007_episode1_timeseries.csv
1000_episode1_timeseries.csv
10013_episode1_timeseries.csv
10017_episode1_timeseries.csv
10027_episode1_timeseries.csv
10028_episode1_timeseries.csv
10029_episode1_timeseries.csv
10032_episode1_timeseries.csv
10038_episode1_timeseries.csv


In [15]:
!pip install mamba-ssm
!pip install einops
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File

In [16]:
import torch
from mamba_ssm import Mamba

print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'mamba_ssm'

In [7]:
!pip install transformers peft scikit-learn accelerate

In [8]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

CLEAN_DIR = "/content/data/in-hospital-mortality-cleaned"
RAW_DIR = "/content/data/in-hospital-mortality"
DRIVE_SAVE_DIR = "/content/drive/MyDrive/icu_llm_data"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

MAX_HOURS = 48

In [18]:
def build_text_from_timeseries(df):
    df = df[df["Hours"] <= MAX_HOURS]

    summary = ["ICU Stay Summary (First 48 Hours):"]

    for col in df.columns:
        if col == "Hours":
            continue

        values = df[col].dropna()
        if len(values) == 0:
            continue

        mean = values.mean()
        std = values.std()
        last = values.iloc[-1]

        summary.append(
            f"{col}: mean {mean:.2f}, standard deviation {std:.2f}, "
            f"last recorded value {last:.2f}."
        )

    return " ".join(summary)

In [19]:
def build_split(split):
    listfile = pd.read_csv(f"{RAW_DIR}/{split}_listfile.csv")
    folder = "train" if split == "val" else split

    texts = []
    labels = []

    for _, row in tqdm(listfile.iterrows(), total=len(listfile)):
        fname = row["stay"]
        label = row["y_true"]

        path = Path(f"{CLEAN_DIR}/{folder}/{fname}")
        df = pd.read_csv(path)

        text = build_text_from_timeseries(df)

        texts.append(text)
        labels.append(label)

    df_out = pd.DataFrame({"text": texts, "label": labels})
    df_out.to_csv(f"{DRIVE_SAVE_DIR}/{split}_text.csv", index=False)

    return df_out

In [20]:
train_df = build_split("train")
val_df   = build_split("val")
test_df  = build_split("test")

print("Done.")
print("Example text length:", len(train_df["text"][0]))

100%|██████████| 3236/3236 [00:14<00:00, 230.47it/s]


Done.
Example text length: 1216


In [21]:
train_df["text"][0][:500]

'ICU Stay Summary (First 48 Hours): Diastolic blood pressure: mean 43.98, standard deviation 4.93, last recorded value 50.00. Fraction inspired oxygen: mean 0.44, standard deviation 0.05, last recorded value 0.50. Glascow coma scale eye opening: mean 3.83, standard deviation 0.39, last recorded value 4.00. Glascow coma scale motor response: mean 6.00, standard deviation 0.00, last recorded value 6.00. Glascow coma scale verbal response: mean 5.00, standard deviation 0.00, last recorded value 5.00'

In [22]:
train_df["text"].str.len().describe()

,text
count,14681.000000
mean,1161.579593
std,120.522685
min,107.000000
25%,1137.000000
50%,1186.000000
75%,1213.000000
max,1388.000000


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

model.to(device)

print("Model loaded.")

KeyboardInterrupt: 

In [26]:
from torch.utils.data import Dataset

MAX_LENGTH = 512

class ICUTextDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [27]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_dataset = ICUTextDataset(train_df)
val_dataset   = ICUTextDataset(val_df)
test_dataset  = ICUTextDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("DataLoaders ready.")

DataLoaders ready.


In [28]:
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# ---- CLASS IMBALANCE FIX ----
pos = train_df["label"].sum()
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos]).to(device)
print("Positive weight:", pos_weight)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

EPOCHS = 3

Positive weight: tensor([6.3885], device='cuda:0', dtype=torch.float64)


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits.view(-1)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Train Loss:", total_loss / len(train_loader))

    # Validation
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits.view(-1)
            probs = torch.sigmoid(logits)

            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    roc = roc_auc_score(all_labels, all_preds)
    pr  = average_precision_score(all_labels, all_preds)

    print(f"Validation ROC-AUC: {roc:.4f}")
    print(f"Validation PR-AUC:  {pr:.4f}")

Epoch 1 Train Loss: 1.0176049701723398
Validation ROC-AUC: 0.7767
Validation PR-AUC:  0.3789
Epoch 2 Train Loss: 1.1096232954031242
Validation ROC-AUC: 0.7710
Validation PR-AUC:  0.3546


In [31]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits.squeeze()
        probs = torch.sigmoid(logits)

        all_preds.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

roc = roc_auc_score(all_labels, all_preds)
pr  = average_precision_score(all_labels, all_preds)

print("Test ROC-AUC:", roc)
print("Test PR-AUC:", pr)

Test ROC-AUC: 0.7821859923691223
Test PR-AUC: 0.32329134445462354


In [32]:
%%writefile clinicalbert_results.txt
Model: ClinicalBERT
Representation: mean + std + last
LR: 2e-5
Batch size: 8
Epochs: 3

Val ROC: 0.7767
Val PR: 0.3789
Test ROC: 0.782
Test PR: 0.323

Writing clinicalbert_results.txt


In [33]:
!mv clinicalbert_results.txt /content/drive/MyDrive/

###----------TRANSFORMER ------

In [12]:
!pip install scikit-learn tqdm

In [13]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLEAN_DIR = "/content/data/in-hospital-mortality-cleaned"
RAW_DIR = "/content/data/in-hospital-mortality"
MAX_HOURS = 48

Device: cpu


In [14]:
def infer_feature_columns():
    sample = next(Path(CLEAN_DIR + "/train").glob("*_timeseries.csv"))
    df = pd.read_csv(sample)
    return sorted([c for c in df.columns if c != "Hours"])

feature_cols = infer_feature_columns()
print("Number of features:", len(feature_cols))

Number of features: 17


In [15]:
def resample_hourly(df, feature_cols):
    df = df.sort_values("Hours")
    df["Hour_int"] = df["Hours"].astype(int)
    df = df[df["Hour_int"] < MAX_HOURS]
    df = df.groupby("Hour_int")[feature_cols].mean()
    df = df.reindex(range(MAX_HOURS))
    return df.to_numpy(dtype=np.float32)

In [16]:
class IHMDataset(Dataset):
    def __init__(self, split, feature_cols, mean=None, std=None):
        self.feature_cols = feature_cols
        self.mean = mean
        self.std = std

        listfile = pd.read_csv(f"{RAW_DIR}/{split}_listfile.csv")
        folder = "train" if split == "val" else split

        self.X = []
        self.y = []

        for _, row in tqdm(listfile.iterrows(), total=len(listfile)):
            fname = row["stay"]
            label = row["y_true"]

            path = Path(f"{CLEAN_DIR}/{folder}/{fname}")
            df = pd.read_csv(path)

            X = resample_hourly(df, feature_cols)

            self.X.append(X)
            self.y.append(label)

        self.X = np.stack(self.X)
        self.y = np.array(self.y)

        if mean is not None:
            self.X = (self.X - mean) / (std + 1e-6)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
            x = self.X[idx]
            x = np.nan_to_num(x)
            return (
                torch.tensor(x, dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32)
            )

In [17]:
train_raw = IHMDataset("train", feature_cols)
train_data = train_raw.X

mean = np.nanmean(train_data, axis=(0,1))
std  = np.nanstd(train_data, axis=(0,1))

train_dataset = IHMDataset("train", feature_cols, mean, std)
val_dataset   = IHMDataset("val", feature_cols, mean, std)
test_dataset  = IHMDataset("test", feature_cols, mean, std)

100%|██████████| 3236/3236 [00:17<00:00, 187.19it/s]


In [18]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [19]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=0.2):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)  # mean pooling over time
        logits = self.classifier(x)
        return logits

In [20]:
# Fix for sympy/torch incompatibility: Uninstall current sympy and install a compatible version.
# *** A Colab runtime restart is required after running this cell once for the changes to take effect. ***
#!pip install sympy==1.12

import torch.nn as nn

model = TimeSeriesTransformer(input_dim=len(feature_cols)).to(device)

pos = train_dataset.y.sum()
neg = len(train_dataset) - pos
pos_weight = torch.tensor([neg / pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 15

In [22]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            logits = model(x).view(-1)
            probs = torch.sigmoid(logits)

            preds.extend(probs.cpu().numpy())
            labels.extend(y.numpy())

    roc = roc_auc_score(labels, preds)
    pr = average_precision_score(labels, preds)

    print(f"Epoch {epoch+1} | Loss {total_loss/len(train_loader):.4f} | ROC {roc:.4f} | PR {pr:.4f}")

Epoch 1 | Loss 1.0040 | ROC 0.8128 | PR 0.4363
Epoch 2 | Loss 0.9478 | ROC 0.8245 | PR 0.4452
Epoch 3 | Loss 0.9300 | ROC 0.8183 | PR 0.4215
Epoch 4 | Loss 0.9286 | ROC 0.8257 | PR 0.4601
Epoch 5 | Loss 0.9064 | ROC 0.8186 | PR 0.4547
Epoch 6 | Loss 0.9202 | ROC 0.8109 | PR 0.4340
Epoch 7 | Loss 0.9052 | ROC 0.8200 | PR 0.4461
Epoch 8 | Loss 0.9004 | ROC 0.8209 | PR 0.4471
Epoch 9 | Loss 0.9007 | ROC 0.8134 | PR 0.4373
Epoch 10 | Loss 0.9200 | ROC 0.8244 | PR 0.4439
Epoch 11 | Loss 0.9152 | ROC 0.8237 | PR 0.4585
Epoch 12 | Loss 0.8993 | ROC 0.8200 | PR 0.4611
Epoch 13 | Loss 0.9066 | ROC 0.8194 | PR 0.4483
Epoch 14 | Loss 0.9071 | ROC 0.8019 | PR 0.3743
Epoch 15 | Loss 0.9168 | ROC 0.7997 | PR 0.3927


In [23]:
model.eval()
preds = []
labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)

        preds.extend(probs.cpu().numpy())
        labels.extend(y.numpy())

print("Test ROC:", roc_auc_score(labels, preds))
print("Test PR:", average_precision_score(labels, preds))

Test ROC: 0.8082769986210608
Test PR: 0.3237020071229633


#---------SHAP--------

In [45]:
!pip install shap

In [46]:
import shap

class TransformerWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # x: [N, D]
        # Expand back to [N, 48, D]
        x = x.unsqueeze(1).repeat(1, 48, 1)
        return torch.sigmoid(self.model(x))

In [47]:
background = []

for i in range(100):
    x, _ = train_dataset[i]
    background.append(x.mean(dim=0).numpy())

background = np.array(background)

In [ ]:
X_explain = []
for i in range(200):
    x, _ = test_dataset[i]
    X_explain.append(x.mean(dim=0).numpy())

X_explain = np.array(X_explain)

In [ ]:
wrapper = TransformerWrapper(model).to(device)
wrapper.eval()

explainer = shap.DeepExplainer(wrapper, torch.tensor(background).to(device))

shap_values = explainer.shap_values(torch.tensor(X_explain).to(device))

In [ ]:
shap.summary_plot(
    shap_values[0],
    X_explain,
    feature_names=feature_cols
)

ClincalBert- Freeze base and train one one head

In [3]:
!pip install -q transformers scikit-learn

In [4]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [6]:
#load model
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [5]:
#load data files
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/icu_llm_data"

train_df = pd.read_csv(f"{BASE_PATH}/train_text.csv")
val_df   = pd.read_csv(f"{BASE_PATH}/val_text.csv")
test_df  = pd.read_csv(f"{BASE_PATH}/test_text.csv")

print(train_df.head())
print(train_df.columns)

                                                text  label
0  ICU Stay Summary (First 48 Hours): Diastolic b...      0
1  ICU Stay Summary (First 48 Hours): Diastolic b...      0
2  ICU Stay Summary (First 48 Hours): Diastolic b...      0
3  ICU Stay Summary (First 48 Hours): Diastolic b...      0
4  ICU Stay Summary (First 48 Hours): Diastolic b...      0
Index(['text', 'label'], dtype='object')


In [22]:
#load dataset class
import torch
from torch.utils.data import Dataset, DataLoader

class IHMTextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()   # <-- FIXED
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [23]:
#recreate dataloaders
BATCH_SIZE = 16

train_dataset = IHMTextDataset(train_df, tokenizer)
val_dataset   = IHMTextDataset(val_df, tokenizer)
test_dataset  = IHMTextDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [24]:
#load model
from transformers import AutoModel
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

bert = AutoModel.from_pretrained(MODEL_NAME)

# Freeze all backbone layers
for param in bert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
#add classification head
class FrozenBERTClassifier(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits.squeeze(-1)

model = FrozenBERTClassifier(bert)
model.to(device)

FrozenBERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [26]:
#optimizer
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-4)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
#training loop
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss / len(train_loader)}")

    # Validation
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits)

            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    roc = roc_auc_score(all_labels, all_preds)
    pr  = average_precision_score(all_labels, all_preds)

    print(f"Validation ROC-AUC: {roc:.4f}")
    print(f"Validation PR-AUC:  {pr:.4f}")

Epoch 1 | Train Loss: 0.4019868078850583
Validation ROC-AUC: 0.5938
Validation PR-AUC:  0.1868


unfreeze last 2 layers and train classifier while keeping rest of the bert frozen

In [6]:
#mount drive, install transformaer, load data steps already completed by re-running above codes
#load tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [7]:
#recreate dataset class
import torch
from torch.utils.data import Dataset, DataLoader

class IHMTextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [8]:
#recreate data loaders
BATCH_SIZE = 16

train_loader = DataLoader(IHMTextDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(IHMTextDataset(val_df, tokenizer), batch_size=BATCH_SIZE)
test_loader  = DataLoader(IHMTextDataset(test_df, tokenizer), batch_size=BATCH_SIZE)

In [9]:
#load bioclinicalBERT again
from transformers import AutoModel
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert = AutoModel.from_pretrained(MODEL_NAME)

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
#freeze all layers first
for param in bert.parameters():
    param.requires_grad = False

In [12]:
#unfreeze last 2 encoder layers
# BioClinicalBERT has 12 layers: 0–11
for param in bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [13]:
# Also unfreeze pooler if present
if hasattr(bert, "pooler"):
    for param in bert.pooler.parameters():
        param.requires_grad = True

In [14]:
#add classification head
class PartialFrozenBERT(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits.squeeze(-1)

model = PartialFrozenBERT(bert)
model.to(device)

PartialFrozenBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [15]:
#optimizer only trainable parameters
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5   # lower LR since we are tuning BERT layers
)

criterion = nn.BCEWithLogitsLoss()

In [16]:
#training loop
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss / len(train_loader)}")

    # Validation
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits)

            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    roc = roc_auc_score(all_labels, all_preds)
    pr  = average_precision_score(all_labels, all_preds)

    print(f"Validation ROC-AUC: {roc:.4f}")
    print(f"Validation PR-AUC:  {pr:.4f}")

Epoch 1 | Train Loss: 0.3638628110825236
Validation ROC-AUC: 0.8026
Validation PR-AUC:  0.4130
Epoch 2 | Train Loss: 0.32424325249228864
Validation ROC-AUC: 0.8273
Validation PR-AUC:  0.4431
Epoch 3 | Train Loss: 0.3148724888583284
Validation ROC-AUC: 0.8318
Validation PR-AUC:  0.4604
Epoch 4 | Train Loss: 0.3066931704631428
Validation ROC-AUC: 0.8358
Validation PR-AUC:  0.4842
Epoch 5 | Train Loss: 0.3019477733145093
Validation ROC-AUC: 0.8338
Validation PR-AUC:  0.4805


In [17]:
#evaluate on test set
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)

        all_preds.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import roc_auc_score, average_precision_score

print("Test ROC:", roc_auc_score(all_labels, all_preds))
print("Test PR :", average_precision_score(all_labels, all_preds))

Test ROC: 0.8460941266157692
Test PR : 0.4417161098004961


Difficult vs easy cases approach using XGBoost

In [1]:
#intsall libraries
!pip install xgboost scikit-learn pandas numpy

In [19]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import xgboost as xgb

In [20]:
#load feature data
DATA_PATH = "/content/drive/MyDrive/in-hospital-mortality-7subseq-features/"

train_df = pd.read_csv(DATA_PATH + "train_features.csv")
val_df   = pd.read_csv(DATA_PATH + "val_features.csv")
test_df  = pd.read_csv(DATA_PATH + "test_features.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14681, 715)
(3222, 715)
(3236, 715)


In [21]:
#spot check to ensure feature columns match across splits
print((train_df.columns == val_df.columns).all())
print((train_df.columns == test_df.columns).all())

True
True


In [22]:
#separate features and labels
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_val = val_df.drop(columns=["label"])
y_val = val_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

In [23]:
#impute missing values; fit imputer only on training data
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_val   = imputer.transform(X_val)
X_test  = imputer.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Height_full_skew' 'Height_first10_std' 'Height_first10_skew'
 'Height_first25_std' 'Height_first25_skew' 'Height_first50_skew'
 'Height_last50_std' 'Height_last50_skew' 'Height_last25_std'
 'Height_last25_skew' 'Height_last10_std' 'Height_last10_skew']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['Height_full_skew' 'Height_first10_std' 'Height_first10_skew'
 'Height_first25_std' 'Height_first25_skew' 'Height_first50_skew'
 'Height_last50_std' 'Height_last50_skew' 'Height_last25_std'
 'Height_last25_skew' 'Height_last10_std' 'Height_last10_skew']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/

In [24]:
#standardize
# tree models technically don't require scaling, but helps consistency for later models
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [25]:
#train XGBoost
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist"
)

In [26]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [27]:
#evaluation on validation set
val_probs = model.predict_proba(X_val)[:,1]

print("Validation ROC:", roc_auc_score(y_val, val_probs))
print("Validation PR:", average_precision_score(y_val, val_probs))

Validation ROC: 0.8089233849456986
Validation PR: 0.4303924762943715


In [28]:
#evaluation on test set
test_probs = model.predict_proba(X_test)[:,1]

print("Test ROC:", roc_auc_score(y_test, test_probs))
print("Test PR:", average_precision_score(y_test, test_probs))

Test ROC: 0.8125931905066199
Test PR: 0.3642734154717615


In [29]:
#training predictions
train_probs = model.predict_proba(X_train)[:,1]

train_preds = (train_probs >= 0.5).astype(int)

train_results = pd.DataFrame({
    "label": y_train,
    "prob": train_probs,
    "pred": train_preds
})

train_results["correct"] = (train_results["label"] == train_results["pred"]).astype(int)

train_results["correct"].value_counts()

,count
correct,
1,14650
0,31


In [30]:
#save xgboost training predictions result
train_results.to_csv(
    "/content/drive/MyDrive/in-hospital-mortality-7subseq-features/xgb_train_predictions.csv",
    index=False
)

In [31]:
#train logistic regression baseline on validation set
#since our training accuracy was 14650 correct and 31 incorrect
#that means our model has memorised the training set
#so use out-of-sample predictions (validation set)
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

lr_model.fit(X_train, y_train)

lr_val_probs = lr_model.predict_proba(X_val)[:,1]
lr_val_preds = (lr_val_probs >= 0.5).astype(int)

In [32]:
#train random forest baseline
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

rf_val_probs = rf_model.predict_proba(X_val)[:,1]
rf_val_preds = (rf_val_probs >= 0.5).astype(int)

In [33]:
#use xgboost validation predictions
xgb_val_probs = model.predict_proba(X_val)[:,1]
xgb_val_preds = (xgb_val_probs >= 0.5).astype(int)

In [34]:
#combine predictions
val_results = pd.DataFrame({
    "label": y_val,

    "lr_pred": lr_val_preds,
    "rf_pred": rf_val_preds,
    "xgb_pred": xgb_val_preds
})

In [35]:
#determine correctness per model
val_results["lr_correct"] = (val_results["lr_pred"] == val_results["label"])
val_results["rf_correct"] = (val_results["rf_pred"] == val_results["label"])
val_results["xgb_correct"] = (val_results["xgb_pred"] == val_results["label"])

In [36]:
#compute misclassification count
val_results["num_models_wrong"] = (
    (~val_results["lr_correct"]).astype(int) +
    (~val_results["rf_correct"]).astype(int) +
    (~val_results["xgb_correct"]).astype(int)
)

#Now each patient has-
#0 → all models correct
#1 → one model wrong
#2 → two models wrong
#3 → all models wrong

In [37]:
#define difficulty
#easy case- num_models_wrong <= 1
#difficult case - num_models_wrong >= 2
val_results["difficulty"] = val_results["num_models_wrong"].apply(
    lambda x: "difficult" if x >= 2 else "easy"
)

In [38]:
# check distribution
val_results["difficulty"].value_counts()

,count
difficulty,
easy,2800
difficult,422


Validation sample- 3222
easy cases- 2800/3222 =. 86.9%
difficult cases- 422/3222 = 13.1%

In [39]:
#save difficulty labels
val_results.to_csv(
    "/content/drive/MyDrive/in-hospital-mortality-7subseq-features/validation_difficulty_labels.csv",
    index=False
)

Near-perfect training accuracy suggests:714 statistical features are extremely expressive

Which is good for modeling but makes training-based difficulty detection unreliable, hence the validation-based approach.


In [40]:
#visualizing difficulty distribution
val_results["num_models_wrong"].value_counts()

,count
num_models_wrong,
0,2077
1,723
2,262
3,160


160 ICU cases are the hardest

In [41]:
#motality rate by difficulty
val_results.groupby("num_models_wrong")["label"].mean()

,label
num_models_wrong,
0,0.009629
1,0.073306
2,0.824427
3,0.918750


mortality by difficulty cases-
easy cases- ~1%- patients almost always survive
medium difficulty- ~7% - still mostly survivors, but slightly harder
difficult cases- ~82% - most patients die
hardest cases- ~92% - extremely high risk patients that all models failed to detect correctly

In [42]:
#check overall mortality of validation set
val_results["label"].mean()

np.float64(0.13531967721911856)

perform multimodal difficulty test using LR, RF, XGBoost

In [3]:
#load feature dataset
import pandas as pd

train_df = pd.read_csv(
"/content/drive/MyDrive/in-hospital-mortality-7subseq-features/train_features.csv"
)

val_df = pd.read_csv(
"/content/drive/MyDrive/in-hospital-mortality-7subseq-features/val_features.csv"
)

test_df = pd.read_csv(
"/content/drive/MyDrive/in-hospital-mortality-7subseq-features/test_features.csv"
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14681, 715)
(3222, 715)
(3236, 715)


In [4]:
#separate features and labels
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_val = val_df.drop(columns=["label"])
y_val = val_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

In [7]:
#mean imputation adopted from paper because subsequences produce NaNs
feature_means = X_train.mean()

X_train = X_train.fillna(feature_means)
X_val = X_val.fillna(feature_means)
X_test = X_test.fillna(feature_means)

# fill columns where mean was NaN
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

In [8]:
#verify; the expected output is 0
print(X_train.isna().sum().sum())

0


In [9]:
X_train.shape

(14681, 714)

In [10]:
#train logistic regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

lr.fit(X_train, y_train)

lr_train_preds = lr.predict(X_train)
lr_val_preds = lr.predict(X_val)

In [11]:
#train random forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

rf_train_preds = rf.predict(X_train)
rf_val_preds = rf.predict(X_val)

In [12]:
#train XGBost
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

xgb_train_preds = xgb.predict(X_train)
xgb_val_preds = xgb.predict(X_val)

In [13]:
#compute model correctness
import numpy as np

val_results = pd.DataFrame({
    "label": y_val,
    "lr_correct": (lr_val_preds == y_val).astype(int),
    "rf_correct": (rf_val_preds == y_val).astype(int),
    "xgb_correct": (xgb_val_preds == y_val).astype(int)
})

In [14]:
#compute difficulty score
val_results["num_models_wrong"] = (
    (1 - val_results["lr_correct"]) +
    (1 - val_results["rf_correct"]) +
    (1 - val_results["xgb_correct"])
)

In [15]:
#define difficulty
val_results["difficulty"] = val_results["num_models_wrong"].apply(
    lambda x: "easy" if x <= 1 else "difficult"
)

In [16]:
#check distribution
val_results["difficulty"].value_counts()

,count
difficulty,
easy,2690
difficult,532


easy- 83.5%
difficult- 16.5%

In [30]:
#to understand extra 110 cases captured as difficult by multi modal framework
val_results[val_results["num_models_wrong"] == 2]

,label,lr_correct,rf_correct,xgb_correct,num_models_wrong,difficulty
10,0,0,0,1,2,difficult
12,0,0,0,1,2,difficult
44,0,0,0,1,2,difficult
46,0,0,0,1,2,difficult
57,0,0,0,1,2,difficult
...,...,...,...,...,...,...
3187,1,1,0,0,2,difficult
3189,0,0,0,1,2,difficult
3193,0,0,0,1,2,difficult
3195,0,0,0,1,2,difficult


In [17]:
#save difficulty prediction on val
val_results.to_csv(
    "/content/drive/MyDrive/in-hospital-mortality-7subseq-features/LR_RF_XGBoost_validation_difficulty_labels.csv",
    index=False
)

In [18]:
#router predicts easy vs difficult
router_labels = (val_results["difficulty"] == "difficult").astype(int)

In [19]:
#train router
router = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss"
)

router.fit(X_val, router_labels)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
#use router on training set
train_diff_probs = router.predict_proba(X_train)[:,1]

train_difficulty = (train_diff_probs >= 0.5).astype(int)

In [21]:
#split training data
X_easy = X_train[train_difficulty == 0]
y_easy = y_train[train_difficulty == 0]

X_difficult = X_train[train_difficulty == 1]
y_difficult = y_train[train_difficulty == 1]

print(X_easy.shape, X_difficult.shape)

(13664, 714) (1017, 714)


In [22]:
#train specialized models
#easy model
easy_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05
)

easy_model.fit(X_easy, y_easy)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
#difficult model example
difficult_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03
)

difficult_model.fit(X_difficult, y_difficult)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [24]:
#router prediction
val_router_probs = router.predict_proba(X_val)[:,1]
val_router_pred = (val_router_probs >= 0.5).astype(int)

0 - easy case
1 - difficult case

In [25]:
#route to specialized model
import numpy as np

final_preds = np.zeros(len(X_val))

easy_idx = val_router_pred == 0
diff_idx = val_router_pred == 1

final_preds[easy_idx] = easy_model.predict_proba(X_val[easy_idx])[:,1]
final_preds[diff_idx] = difficult_model.predict_proba(X_val[diff_idx])[:,1]

In [26]:
#compute metrics
from sklearn.metrics import roc_auc_score, average_precision_score

roc = roc_auc_score(y_val, final_preds)
pr = average_precision_score(y_val, final_preds)

print("Routing Model ROC:", roc)
print("Routing Model PR :", pr)

Routing Model ROC: 0.8402670297753514
Routing Model PR : 0.43234809404069474


In [27]:
#compare aginst single XGBoost
xgb_val_probs = xgb.predict_proba(X_val)[:,1]

from sklearn.metrics import roc_auc_score, average_precision_score

print("Baseline XGBoost ROC:", roc_auc_score(y_val, xgb_val_probs))
print("Baseline XGBoost PR :", average_precision_score(y_val, xgb_val_probs))

Baseline XGBoost ROC: 0.8049059188471848
Baseline XGBoost PR : 0.4160479313501426


In [28]:
#feature importance by difficulty
easy_model.feature_importances_
difficult_model.feature_importances_

array([0.0000000e+00, 0.0000000e+00, 2.7051859e-03, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 1.6666763e-03, 1.4973587e-03,
       1.2430371e-03, 1.4529815e-03, 1.0706389e-03, 2.1403981e-03,
       1.0026902e-03, 3.0358750e-03, 2.9780793e-03, 2.0273561e-03,
       1.4981866e-03, 1.4550112e-03, 1.5173008e-03, 1.1629120e-03,
       1.1717499e-03, 1.7791102e-03, 9.3409081e-04, 7.8062719e

In [29]:
#check how router split validation patients
np.bincount(val_router_pred)

array([2786,  436])

In [31]:
#check router accuracy
#tells us how well router identifies difficult patients
from sklearn.metrics import accuracy_score

print("Router Accuracy:", accuracy_score(router_labels, val_router_pred))

Router Accuracy: 0.9695841092489137


In [32]:
#evaluate routing system on test set to confirm generalization
test_router_probs = router.predict_proba(X_test)[:,1]
test_router_pred = (test_router_probs >= 0.5).astype(int)

final_test_preds = np.zeros(len(X_test))

easy_idx = test_router_pred == 0
diff_idx = test_router_pred == 1

final_test_preds[easy_idx] = easy_model.predict_proba(X_test[easy_idx])[:,1]
final_test_preds[diff_idx] = difficult_model.predict_proba(X_test[diff_idx])[:,1]

print("Routing Test ROC:", roc_auc_score(y_test, final_test_preds))
print("Routing Test PR :", average_precision_score(y_test, final_test_preds))

Routing Test ROC: 0.8033628927080648
Routing Test PR : 0.35040950737673937


In [33]:
#feature imporatnce between easy and difficult models
import pandas as pd

easy_importance = pd.Series(
easy_model.feature_importances_,
index=X_train.columns
).sort_values(ascending=False)

diff_importance = pd.Series(
difficult_model.feature_importances_,
index=X_train.columns
).sort_values(ascending=False)

In [34]:
easy_importance.head(10)
diff_importance.head(10)

,0
Temperature_first10_skew,0.010395
Temperature_first10_std,0.008330
Glascow coma scale total_last50_mean,0.007850
Weight_last50_count,0.007691
Glascow coma scale motor response_last25_std,0.007415
Glascow coma scale motor response_first50_count,0.007376
Glascow coma scale total_last10_max,0.006952
Glascow coma scale verbal response_full_skew,0.006527
Weight_first25_min,0.006148
pH_last50_mean,0.006022
